# RSNA Knee Abnormality Detection — Experiment Notebook

**Purpose:** keep the reproducible experiments in one ordered notebook and remove exploratory/debugging clutter.

## How to use this notebook

- **GPU OFF:** dataset-label audit, report inspection, logistic-regression experiments, and weak-label generation/audit.
- **GPU ON:** only when running the DINOv2 feature-extraction cell (EXP-004). It explicitly requires CUDA.
- Do **not** use **Run All** by default. Run the dataset audit first. Run feature extraction only if the cache is missing; otherwise skip it and validate the cache.
- The cache and weak-label outputs live under `/kaggle/working/` and are temporary runtime files. Download important outputs before ending the session.
- The fixed gold-label split is `SPLIT-42-ACL-43-15`: 43 train studies / 15 validation studies, seed 42, stratified by ACL.
- Only 58 studies have all 12 official labels. The other 4,349 studies have all targets missing; missing labels are **not negatives**.
- Validation metrics are highly uncertain because the labelled validation set has only 15 studies. Treat small differences as exploratory, not conclusive.

## 1. Dataset and label audit

Run with **GPU OFF**. This verifies the central label-missingness fact before experiments.

In [ ]:

import pandas as pd
from pathlib import Path

DATA = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
df = pd.read_csv(DATA / "train.csv")

targets = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

print("Studies:", len(df))
print("\nObserved labels per target:")
print(df[targets].notna().sum().to_string())

print("\nMissing labels per target:")
print(df[targets].isna().sum().to_string())

observed_per_study = df[targets].notna().sum(axis=1)
print("\nObserved labels per study:")
print(observed_per_study.value_counts().sort_index().to_string())

print("\nStudies with at least one observed target:",
      int((observed_per_study > 0).sum()))
print("Studies with all 12 targets observed:",
      int((observed_per_study == 12).sum()))

print("\nLabel value counts, including missing values:")
for target in targets:
    print(f"\n{target}")
    print(df[target].value_counts(dropna=False).to_string())


## 2. Historical baseline results

These are recorded results from the earlier notebook. This section is a record only; it does not rerun the old CNNs.

| Experiment | Method | Validation macro-AUC |
|---|---|---:|
| EXP-001 | Three-slice CNN, slice predictions averaged per study | 0.5018 |
| EXP-002 | Study-level CNN, 3 slices as channels | 0.4628 |
| EXP-003 | Study-level CNN, 9 slices as channels | 0.5684 |
| EXP-004 | Frozen DINOv2 Base embeddings + logistic regression | 0.5717 |
| EXP-005 | Report text TF-IDF + logistic regression | 0.5351 |
| EXP-006 | DINOv2 embeddings + report TF-IDF | 0.5665 |

**Interpretation:** EXP-004 is the highest of these recorded baselines, but the validation sample is too small to conclude that the difference is robust. Do not compare these local results directly with leaderboard scores without checking evaluation protocols.

## 3. EXP-004 — DINOv2 feature extraction

**GPU ON.** This cell extracts features for the 58 fully labelled studies and writes a cache to `/kaggle/working/exp004_dinov2_cache`. Skip this cell if that cache already exists and is complete.

In [ ]:

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import pydicom
import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModel

# ---------------- Configuration ----------------
DATA = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
MODEL_PATH = "/kaggle/input/models/metaresearch/dinov2/pytorch/base/1"
OUT = Path("/kaggle/working/exp004_dinov2_cache")
OUT.mkdir(parents=True, exist_ok=True)

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]
N_SLICES = 9
SEED = 42

# ---------------- Data and split ----------------
labels = pd.read_csv(DATA / "train.csv")
series_df = pd.read_csv(DATA / "train_series.csv")

labelled = labels.dropna(subset=TARGETS).copy()
labelled = labelled.sort_values("StudyInstanceUID").reset_index(drop=True)

from sklearn.model_selection import train_test_split

train_ids, val_ids = train_test_split(
    labelled["StudyInstanceUID"].tolist(),
    test_size=0.25,
    random_state=SEED,
    stratify=labelled["ACL"]
)
split = {study: "train" for study in train_ids}
split.update({study: "validation" for study in val_ids})

assert len(labelled) == 58
assert len(train_ids) == 43 and len(val_ids) == 15

# ---------------- Model ----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("GPU is not enabled. Turn the Kaggle accelerator on.")

processor = AutoImageProcessor.from_pretrained(
    MODEL_PATH, local_files_only=True
)
model = AutoModel.from_pretrained(
    MODEL_PATH, local_files_only=True
).to(device).eval()

print("Device:", device)
print("Studies:", len(labelled), "| train:", len(train_ids), "| validation:", len(val_ids))

# ---------------- Series selection ----------------
def choose_series(study_id):
    candidates = series_df[series_df["StudyInstanceUID"] == study_id]
    sagittal = candidates[
        (candidates["Anatomical_Plane"] == "Sagittal")
        & (candidates["Fluid_Sensitive"] == 1)
    ]
    coronal = candidates[
        (candidates["Anatomical_Plane"] == "Coronal")
        & (candidates["Fluid_Sensitive"] == 1)
    ]

    if not sagittal.empty:
        return sagittal.sort_values("SeriesInstanceUID", kind="stable").iloc[0], "sagittal_fluid"
    if not coronal.empty:
        return coronal.sort_values("SeriesInstanceUID", kind="stable").iloc[0], "coronal_fluid_fallback"

    raise RuntimeError(f"No eligible fluid-sensitive series for {study_id}")

def load_ordered_dicom_paths(study_id, series_id):
    folder = DATA / "train_series" / str(study_id) / str(series_id)
    paths = list(folder.glob("*.dcm"))
    if not paths:
        raise FileNotFoundError(f"No DICOM files in {folder}")

    # Read metadata and order slices deterministically.
    entries = []
    for path in paths:
        ds = pydicom.dcmread(path, stop_before_pixels=True)
        instance = getattr(ds, "InstanceNumber", None)
        if instance is None:
            raise ValueError(f"Missing InstanceNumber: {path}")
        entries.append((int(instance), path))

    entries.sort(key=lambda x: (x[0], x[1].name))
    instances = [x[0] for x in entries]
    if len(instances) != len(set(instances)):
        raise ValueError(f"Duplicate InstanceNumber in {study_id}/{series_id}")

    return [x[1] for x in entries]

def dicom_to_image(path):
    ds = pydicom.dcmread(path)
    pixels = ds.pixel_array.astype(np.float32)
    pixels = pixels * float(getattr(ds, "RescaleSlope", 1))
    pixels += float(getattr(ds, "RescaleIntercept", 0))

    lo, hi = np.percentile(pixels, [1, 99])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        raise ValueError(f"Invalid intensity range: {path}")

    pixels = np.clip((pixels - lo) / (hi - lo), 0, 1)
    pixels = (pixels * 255).astype(np.uint8)

    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        pixels = 255 - pixels

    return Image.fromarray(pixels).convert("RGB"), ds

# ---------------- Extract features ----------------
all_features = []
metadata = []
start = time.time()

for study_idx, row in labelled.iterrows():
    study_id = row["StudyInstanceUID"]
    chosen, selection = choose_series(study_id)
    series_id = chosen["SeriesInstanceUID"]

    paths = load_ordered_dicom_paths(study_id, series_id)
    # Nine deterministic, evenly spaced indices, including both ends.
    indices = np.rint(np.linspace(0, len(paths) - 1, N_SLICES)).astype(int)
    if len(set(indices.tolist())) != N_SLICES:
        raise RuntimeError(f"Repeated slice indices for {study_id}")

    images = []
    slice_metadata = []

    for idx in indices:
        path = paths[int(idx)]
        image, ds = dicom_to_image(path)
        images.append(image)
        slice_metadata.append({
            "study_id": study_id,
            "series_id": series_id,
            "selection": selection,
            "slice_index_in_sorted_series": int(idx),
            "instance_number": int(ds.InstanceNumber),
            "dicom_filename": path.name,
            "split": split[study_id],
        })

    # Process one study (9 slices) at a time to limit GPU memory.
    inputs = processor(images=images, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model(**inputs)
        # CLS token: one 768-dimensional vector per slice.
        feats = outputs.last_hidden_state[:, 0, :]

    feats = feats.float().cpu().numpy()
    if feats.shape != (N_SLICES, 768) or not np.isfinite(feats).all():
        raise RuntimeError(f"Unexpected/invalid features for {study_id}: {feats.shape}")

    all_features.append(feats)
    metadata.extend(slice_metadata)

    if (study_idx + 1) % 5 == 0 or study_idx + 1 == len(labelled):
        print(
            f"{study_idx + 1}/{len(labelled)} studies | "
            f"{time.time() - start:.1f}s elapsed"
        )

# Shape: studies x slices x embedding dimensions
features = np.stack(all_features).astype(np.float32)
study_ids = labelled["StudyInstanceUID"].tolist()

np.save(OUT / "features.npy", features)
np.save(OUT / "study_ids.npy", np.asarray(study_ids))
pd.DataFrame(metadata).to_csv(OUT / "slice_metadata.csv", index=False)

# Labels and split manifest are saved separately for traceability.
labelled[["StudyInstanceUID"] + TARGETS].to_csv(OUT / "labels.csv", index=False)
pd.DataFrame({
    "StudyInstanceUID": study_ids,
    "split": [split[s] for s in study_ids],
}).to_csv(OUT / "split_manifest.csv", index=False)

manifest = {
    "experiment": "EXP-004",
    "model": "DINOv2 Base",
    "model_path": MODEL_PATH,
    "n_studies": len(study_ids),
    "n_slices_per_study": N_SLICES,
    "feature_dim": int(features.shape[-1]),
    "feature_shape": list(features.shape),
    "split_seed": SEED,
    "train_studies": len(train_ids),
    "validation_studies": len(val_ids),
    "selection_policy": "fluid-sensitive sagittal; coronal fallback",
    "slice_sampling": "9 rounded evenly spaced indices after InstanceNumber ordering",
    "preprocessing": "per-slice 1st/99th percentile scaling, MONOCHROME1 inversion, RGB",
    "feature": "DINOv2 last_hidden_state CLS token",
    "elapsed_seconds": round(time.time() - start, 2),
}

with open(OUT / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("\nExtraction complete.")
print("Feature shape:", features.shape)
print("Feature cache:", OUT)
print("Finite values:", bool(np.isfinite(features).all()))
print("Elapsed seconds:", manifest["elapsed_seconds"])


## 4. Verify the DINOv2 cache

**GPU OFF.** Run this before any cached-feature experiment. If the cache is absent, return to EXP-004 and run feature extraction with GPU ON.

In [ ]:

from pathlib import Path
import json
import numpy as np
import pandas as pd

CACHE = Path("/kaggle/working/exp004_dinov2_cache")

features = np.load(CACHE / "features.npy")
study_ids = np.load(CACHE / "study_ids.npy", allow_pickle=True)
labels = pd.read_csv(CACHE / "labels.csv")
split = pd.read_csv(CACHE / "split_manifest.csv")
slice_metadata = pd.read_csv(CACHE / "slice_metadata.csv")
manifest = json.loads((CACHE / "manifest.json").read_text())

print("Features:", features.shape)
print("Studies:", len(study_ids))
print("Unique study IDs:", len(set(study_ids)))
print("Labels:", labels.shape)
print("Split rows:", len(split))
print("Slice metadata rows:", len(slice_metadata))
print("Finite features:", np.isfinite(features).all())
print("Train studies:", (split["split"] == "train").sum())
print("Validation studies:", (split["split"] == "validation").sum())
print("Cache files:", sorted(p.name for p in CACHE.iterdir()))
print("Manifest:", manifest)


## 5. EXP-004 — Image-only classifier

**GPU OFF.** Fits logistic regression on cached DINOv2 study embeddings using the fixed split.

In [ ]:

import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from pathlib import Path

CACHE = Path("/kaggle/working/exp004_dinov2_cache")
TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

features = np.load(CACHE / "features.npy")
study_ids = np.load(CACHE / "study_ids.npy", allow_pickle=True).astype(str)
labels = pd.read_csv(CACHE / "labels.csv")
split = pd.read_csv(CACHE / "split_manifest.csv")

# One feature vector per study: average the nine slice embeddings.
study_features = features.mean(axis=1)
feature_df = pd.DataFrame(study_features)
feature_df.insert(0, "StudyInstanceUID", study_ids)

# Merge by study ID rather than relying on row order.
data = (
    split.merge(labels, on="StudyInstanceUID", validate="one_to_one")
         .merge(feature_df, on="StudyInstanceUID", validate="one_to_one")
)

feature_cols = list(range(768))
train_mask = data["split"].eq("train").to_numpy()
val_mask = data["split"].eq("validation").to_numpy()

assert train_mask.sum() == 43 and val_mask.sum() == 15
assert data["StudyInstanceUID"].is_unique

results = []

for target in TARGETS:
    # Missing labels are excluded, never converted to zero.
    y = pd.to_numeric(data[target], errors="coerce").to_numpy()
    known = np.isfinite(y)
    train_idx = train_mask & known
    val_idx = val_mask & known

    y_train = y[train_idx].astype(int)
    y_val = y[val_idx].astype(int)

    # Logistic regression requires both classes in training;
    # AUC requires both classes in validation.
    if len(y_train) == 0 or len(np.unique(y_train)) < 2:
        print(f"{target}: SKIPPED — training labels lack both classes")
        continue

    if len(y_val) == 0 or len(np.unique(y_val)) < 2:
        print(f"{target}: AUC unavailable — validation labels lack both classes")
        continue

    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=1.0,
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        )
    )

    model.fit(data.loc[train_idx, feature_cols], y_train)
    probabilities = model.predict_proba(
        data.loc[val_idx, feature_cols]
    )[:, 1]

    auc = roc_auc_score(y_val, probabilities)

    results.append({
        "target": target,
        "train_known": int(train_idx.sum()),
        "validation_known": int(val_idx.sum()),
        "validation_positive": int(y_val.sum()),
        "validation_negative": int((y_val == 0).sum()),
        "auc": float(auc)
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

if not results_df.empty:
    macro_auc = results_df["auc"].mean()
    print(f"\nEXP-004 macro-AUC across evaluable targets: {macro_auc:.4f}")
    print("EXP-003 reference macro-AUC: 0.5684")
    print(f"Difference: {macro_auc - 0.5684:+.4f}")

results_df.to_csv(CACHE / "exp004_logistic_results.csv", index=False)
print(f"\nResults saved to {CACHE / 'exp004_logistic_results.csv'}")


## 6. Inspect reports

**GPU OFF.** Displays a small sample of reports to support careful development of report-derived labels. This is an inspection step, not model training.

In [ ]:

from pathlib import Path
import pandas as pd

DATA = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
train = pd.read_csv(DATA / "train.csv")

print("Dataset shape:", train.shape)
print("Columns:", train.columns.tolist())
print("\nMissing reports:", train["Report"].isna().sum())
print("Empty reports:", train["Report"].fillna("").str.strip().eq("").sum())

# Inspect a few reports without changing the data.
for i, row in train[["StudyInstanceUID", "Report"]].head(5).iterrows():
    report = "" if pd.isna(row["Report"]) else str(row["Report"])
    print(f"\n{'=' * 70}")
    print("Study:", row["StudyInstanceUID"])
    print("Report length:", len(report))
    print(report[:2500])


## 7. EXP-005 — Report-text baseline

**GPU OFF.** Fits TF-IDF on training reports only, then evaluates on the same fixed validation split.

In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

DATA = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
CACHE = Path("/kaggle/working/exp004_dinov2_cache")

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

# Reuse the exact study split from EXP-004.
labels = pd.read_csv(CACHE / "labels.csv")
split = pd.read_csv(CACHE / "split_manifest.csv")
train = pd.read_csv(DATA / "train.csv")

data = (
    split.merge(train[["StudyInstanceUID", "Report"]], on="StudyInstanceUID",
                validate="one_to_one")
         .merge(labels, on="StudyInstanceUID", validate="one_to_one")
)

train_mask = data["split"].eq("train").to_numpy()
val_mask = data["split"].eq("validation").to_numpy()

assert train_mask.sum() == 43 and val_mask.sum() == 15
assert data["StudyInstanceUID"].is_unique
assert data["Report"].notna().all()

# Fit TF-IDF on training reports only to prevent vocabulary leakage.
vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=1,
    max_features=20000,
    sublinear_tf=True
)

X_train_all = vectorizer.fit_transform(data.loc[train_mask, "Report"])
X_val_all = vectorizer.transform(data.loc[val_mask, "Report"])

results = []

for target in TARGETS:
    y = pd.to_numeric(data[target], errors="coerce").to_numpy()
    known = np.isfinite(y)
    train_idx = train_mask & known
    val_idx = val_mask & known

    y_train = y[train_idx].astype(int)
    y_val = y[val_idx].astype(int)

    if len(np.unique(y_train)) < 2:
        print(f"{target}: SKIPPED — training labels lack both classes")
        continue

    if len(y_val) == 0 or len(np.unique(y_val)) < 2:
        print(f"{target}: AUC unavailable — validation lacks both classes")
        continue

    # Select the matching rows from the train-only and validation matrices.
    train_positions = np.flatnonzero(train_idx[train_mask])
    val_positions = np.flatnonzero(val_idx[val_mask])

    model = LogisticRegression(
        C=1.0,
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )
    model.fit(X_train_all[train_positions], y_train)
    probabilities = model.predict_proba(X_val_all[val_positions])[:, 1]
    auc = roc_auc_score(y_val, probabilities)

    results.append({
        "target": target,
        "train_known": int(train_idx.sum()),
        "validation_known": int(val_idx.sum()),
        "validation_positive": int(y_val.sum()),
        "validation_negative": int((y_val == 0).sum()),
        "auc": float(auc)
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

if not results_df.empty:
    macro_auc = results_df["auc"].mean()
    print(f"\nEXP-005 text macro-AUC: {macro_auc:.4f}")
    print("EXP-003 image baseline: 0.5684")
    print("EXP-004 DINOv2 image baseline: 0.5717")
    print(f"Difference vs EXP-004: {macro_auc - 0.5717:+.4f}")

results_df.to_csv(CACHE / "exp005_text_results.csv", index=False)
print(f"\nSaved results to {CACHE / 'exp005_text_results.csv'}")


## 8. EXP-006 — Image + report-text baseline

**GPU OFF.** Combines the image embeddings and report text. Treat this as exploratory given the very small labelled validation set.

In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

DATA = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
CACHE = Path("/kaggle/working/exp004_dinov2_cache")

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

# Load cached image embeddings and fixed split.
features = np.load(CACHE / "features.npy").mean(axis=1)
study_ids = np.load(CACHE / "study_ids.npy", allow_pickle=True).astype(str)
labels = pd.read_csv(CACHE / "labels.csv")
split = pd.read_csv(CACHE / "split_manifest.csv")
reports = pd.read_csv(DATA / "train.csv")[["StudyInstanceUID", "Report"]]

image_df = pd.DataFrame(features, columns=[f"img_{i}" for i in range(768)])
image_df.insert(0, "StudyInstanceUID", study_ids)

data = (
    split.merge(labels, on="StudyInstanceUID", validate="one_to_one")
         .merge(reports, on="StudyInstanceUID", validate="one_to_one")
         .merge(image_df, on="StudyInstanceUID", validate="one_to_one")
)

train_mask = data["split"].eq("train").to_numpy()
val_mask = data["split"].eq("validation").to_numpy()

assert train_mask.sum() == 43 and val_mask.sum() == 15
assert data["StudyInstanceUID"].is_unique

# Fit text vocabulary using training reports only.
vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=1,
    max_features=20000,
    sublinear_tf=True
)

X_text_train = vectorizer.fit_transform(data.loc[train_mask, "Report"])
X_text_val = vectorizer.transform(data.loc[val_mask, "Report"])

# Standardise image features using training studies only.
scaler = StandardScaler()
X_img_train = scaler.fit_transform(
    data.loc[train_mask, [f"img_{i}" for i in range(768)]]
)
X_img_val = scaler.transform(
    data.loc[val_mask, [f"img_{i}" for i in range(768)]]
)

# Combine the two modalities.
from scipy.sparse import csr_matrix, hstack

X_train = hstack([csr_matrix(X_img_train), X_text_train]).tocsr()
X_val = hstack([csr_matrix(X_img_val), X_text_val]).tocsr()

results = []

for target in TARGETS:
    y = pd.to_numeric(data[target], errors="coerce").to_numpy()
    known = np.isfinite(y)
    train_idx = train_mask & known
    val_idx = val_mask & known

    y_train = y[train_idx].astype(int)
    y_val = y[val_idx].astype(int)

    if len(np.unique(y_train)) < 2:
        print(f"{target}: skipped — training labels lack both classes")
        continue
    if len(y_val) == 0 or len(np.unique(y_val)) < 2:
        print(f"{target}: AUC unavailable — validation lacks both classes")
        continue

    train_positions = np.flatnonzero(train_idx[train_mask])
    val_positions = np.flatnonzero(val_idx[val_mask])

    model = LogisticRegression(
        C=0.1,
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )
    model.fit(X_train[train_positions], y_train)
    probabilities = model.predict_proba(X_val[val_positions])[:, 1]

    results.append({
        "target": target,
        "validation_positive": int(y_val.sum()),
        "validation_negative": int((y_val == 0).sum()),
        "auc": float(roc_auc_score(y_val, probabilities))
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

if not results_df.empty:
    macro_auc = results_df["auc"].mean()
    print(f"\nEXP-006 image + text macro-AUC: {macro_auc:.4f}")
    print(f"EXP-004 image-only: 0.5717")
    print(f"EXP-005 text-only: 0.5351")
    print(f"Difference vs image-only: {macro_auc - 0.5717:+.4f}")

results_df.to_csv(CACHE / "exp006_combined_results.csv", index=False)
print(f"\nSaved to {CACHE / 'exp006_combined_results.csv'}")


## 9. EXP-007 — Candidate weak-label generation

**GPU OFF.** This self-contained cell contains the current rule-based candidate-label script. It excludes all 58 fully labelled studies and does not treat silence as a negative. Outputs are written to `/kaggle/working/exp007_weak_labels/`.

**Important:** generating labels is not validating them. Inspect `weak_label_audit.csv` and `weak_label_summary.csv` before using any weak labels for training. In the earlier run, several targets had very imbalanced candidate coverage, and some targets had almost no candidate negatives.

In [ ]:
#!/usr/bin/env python3
"""Conservative rule-based weak labels from knee MRI reports.

Only studies with every official target missing are processed. Explicit positive
or negative cues may yield 1 or 0; no cue, uncertainty, or conflicting cues yield
NaN. This is a candidate generator, not validated clinical NLP.
"""
import argparse
import re
from pathlib import Path

import numpy as np
import pandas as pd

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]

# Keep these rules conservative. Silence is never a negative label.
RULES = {
"ACL": {
"positive": [
r"\b(?:acl|anterior cruciate ligament)\b.{0,40}\b(?:tear|torn|rupture|ruptured|sprain|injury)\b",
r"\b(?:tear|rupture|sprain|injury)\b.{0,40}\b(?:acl|anterior cruciate ligament)\b",
r"\b(?:rotura|ruptura)\b.{0,40}\bligamento cruzado anterior\b",
r"\b(?:rupture|déchirure)\b.{0,40}\bligament croisé antérieur\b",
r"\bvoorste kruisband\b.{0,30}\b(?:scheur|ruptuur|letsel)\b"],
"negative": [
r"\b(?:acl|anterior cruciate ligament)\b.{0,25}\b(?:intact|normal|unremarkable)\b",
r"\b(?:no|without|absence of)\b.{0,40}\b(?:acl|anterior cruciate ligament)\b.{0,25}\b(?:tear|rupture|injury)\b",
r"\bligamento cruzado anterior\b.{0,30}\b(?:íntegro|integro|normal)\b",
r"\bligament croisé antérieur\b.{0,30}\b(?:intact|normal)\b"]},
"MCL": {
"positive": [
r"\b(?:mcl|medial collateral ligament)\b.{0,40}\b(?:tear|rupture|sprain|injury|thickening)\b",
r"\b(?:tear|rupture|sprain|injury)\b.{0,40}\b(?:mcl|medial collateral ligament)\b",
r"\b(?:esguince|rotura|ruptura)\b.{0,35}\bligamento colateral medial\b",
r"\b(?:entorse|rupture|déchirure)\b.{0,35}\bligament collatéral médial\b"],
"negative": [
r"\b(?:mcl|medial collateral ligament)\b.{0,25}\b(?:intact|normal|unremarkable)\b",
r"\b(?:no|without|absence of)\b.{0,40}\b(?:mcl|medial collateral ligament)\b.{0,25}\b(?:tear|rupture|injury)\b",
r"\bligamento colateral medial\b.{0,25}\b(?:íntegro|integro|normal)\b"]},
"Medial Meniscus": {
"positive": [
r"\bmedial menisc(?:us|al)\b.{0,40}\b(?:tear|torn|rupture|flap)\b",
r"\b(?:tear|torn|rupture)\b.{0,40}\bmedial menisc(?:us|al)\b",
r"\b(?:rotura|ruptura)\b.{0,35}\bmenisco (?:medial|interno)\b",
r"\b(?:déchirure|rupture)\b.{0,35}\b(?:ménisque médial|ménisque interne)\b",
r"\b(?:mediale meniscus|binnenmeniscus)\b.{0,30}\b(?:scheur|ruptuur)\b"],
"negative": [
r"\bmedial menisc(?:us|al)\b.{0,30}\b(?:intact|normal|not torn|no tear|without tear)\b",
r"\b(?:no|without|absence of|no evidence of|no signs of)\b.{0,35}\b(?:tear|rupture)\b.{0,30}\bmedial menisc(?:us|al)\b",
r"\bmenisco (?:medial|interno)\b.{0,30}\b(?:sin rotura|normal|conservado)\b",
r"\b(?:ménisque médial|ménisque interne)\b.{0,30}\b(?:intact|normal|sans déchirure)\b"]},
"Lateral Meniscus": {
"positive": [
r"\blateral menisc(?:us|al)\b.{0,40}\b(?:tear|torn|rupture|flap)\b",
r"\b(?:tear|torn|rupture)\b.{0,40}\blateral menisc(?:us|al)\b",
r"\b(?:rotura|ruptura)\b.{0,35}\bmenisco lateral\b",
r"\b(?:déchirure|rupture)\b.{0,35}\b(?:ménisque latéral|ménisque externe)\b",
r"\b(?:laterale meniscus|buitenmeniscus)\b.{0,30}\b(?:scheur|ruptuur)\b"],
"negative": [
r"\blateral menisc(?:us|al)\b.{0,30}\b(?:intact|normal|not torn|no tear|without tear)\b",
r"\b(?:no|without|absence of|no evidence of|no signs of)\b.{0,35}\b(?:tear|rupture)\b.{0,30}\blateral menisc(?:us|al)\b",
r"\bmenisco lateral\b.{0,30}\b(?:sin rotura|normal|conservado)\b",
r"\b(?:ménisque latéral|ménisque externe)\b.{0,30}\b(?:intact|normal|sans déchirure)\b"]},
"Medial OA": {
"positive": [
r"\bmedial\b.{0,35}\b(?:osteoarthritis|osteoarthrosis|arthrosis|artrosis|arthrose|gonarthrosis)\b",
r"\b(?:osteoarthritis|osteoarthrosis|arthrosis|artrosis|arthrose|gonarthrosis)\b.{0,35}\bmedial\b",
r"\bartrosis femorotibial medial\b",
r"\bcompartiment médial\b.{0,35}\b(?:arthrose|ostéoarthrose)\b"],
"negative": [
r"\bno\b.{0,25}\bmedial (?:compartment )?(?:osteoarthritis|osteoarthrosis|arthrosis)\b",
r"\bpas d['’]arthrose\b.{0,35}\b(?:compartiment médial|médial)\b"]},
"Lateral OA": {
"positive": [
r"\blateral\b.{0,35}\b(?:osteoarthritis|osteoarthrosis|arthrosis|artrosis|arthrose|gonarthrosis)\b",
r"\b(?:osteoarthritis|osteoarthrosis|arthrosis|artrosis|arthrose|gonarthrosis)\b.{0,35}\blateral\b",
r"\bartrosis femorotibial lateral\b",
r"\bcompartiment latéral\b.{0,35}\b(?:arthrose|ostéoarthrose)\b"],
"negative": [
r"\bno\b.{0,25}\blateral (?:compartment )?(?:osteoarthritis|osteoarthrosis|arthrosis)\b",
r"\bpas d['’]arthrose\b.{0,35}\b(?:compartiment latéral|latéral)\b"]},
"PF OA": {
"positive": [
r"\bpatellofemoral\b.{0,35}\b(?:osteoarthritis|osteoarthrosis|arthrosis|artrosis|arthrose)\b",
r"\b(?:osteoarthritis|osteoarthrosis|arthrosis|artrosis|arthrose)\b.{0,35}\bpatellofemoral\b",
r"\b(?:retropatellar|patellofemoral)\b.{0,35}\b(?:arthrose|artrosis|osteoarthritis)\b"],
"negative": [
r"\bno\b.{0,25}\bpatellofemoral (?:osteoarthritis|osteoarthrosis|arthrosis)\b",
r"\bpas d['’]arthrose\b.{0,35}\b(?:patellofémoral|rétropatellaire)\b"]},
"Effusion": {
"positive": [r"\b(?:joint |knee |articular )?effusion\b",
r"\b(?:derrame articular|derrame de rodilla|épanchement articulaire|épanchement du genou|gewrichtseffusie)\b"],
"negative": [r"\b(?:no|without|absence of|no significant)\b.{0,25}\b(?:joint |knee |articular )?effusion\b",
r"\bno hay (?:evidente )?derrame(?: articular)?\b",
r"\bpas d['’]épanchement(?: articulaire)?\b", r"\bgeen (?:gewrichts)?effusie\b", r"\bkein gelenkerguss\b"]},
"Synovitis": {
"positive": [r"\bsynovitis\b", r"\bsinovitis\b", r"\bsynovite\b", r"\bsynoviale ontsteking\b"],
"negative": [r"\b(?:no|without|absence of)\b.{0,20}\bsynovitis\b", r"\bno hay (?:signos de )?sinovitis\b",
r"\bsin sinovitis\b", r"\bpas de synovite\b", r"\bgeen synovitis\b"]},
"Baker's": {
"positive": [r"\bbaker['’]?s? cyst\b", r"\bpopliteal cyst\b", r"\bquiste de baker\b", r"\bkyste poplité\b", r"\bcyste poplité\b", r"\bbakerzyste\b"],
"negative": [r"\b(?:no|without|absence of)\b.{0,25}\b(?:baker['’]?s? cyst|popliteal cyst)\b",
r"\bno hay (?:quiste de baker|quistes poplíteos?)\b", r"\bpas de kyste poplité\b"]},
"Contusion": {
"positive": [r"\bbone contusion\b", r"\bbone bruise\b", r"\bcontusión ósea\b", r"\bcontusion osseuse\b", r"\bknochenprellung\b"],
"negative": [r"\b(?:no|without|absence of)\b.{0,25}\b(?:bone contusion|bone bruise)\b",
r"\bno hay contusión ósea\b", r"\bpas de contusion osseuse\b", r"\bkeine knochenprellung\b"]},
"Fracture": {
"positive": [r"\bfracture\b", r"\bfractura\b", r"\bfractuur\b", r"\bfraktur\b"],
"negative": [r"\b(?:no|without|absence of|no evidence of)\b.{0,25}\bfracture\b",
r"\b(?:fractures?|fracturas?)\s*:\s*(?:none|no|aucune|ninguna)\b",
r"\bno hay (?:evidencia de )?fractura\b", r"\bpas de fracture\b", r"\bkeine fraktur\b", r"\bgeen fractuur\b"]},
}

UNCERTAINTY = re.compile(
    r"\b(?:possible|possibly|probable|probably|suspected|suspicious for|cannot exclude|"
    r"may represent|might represent|questionable|indeterminate|posible|posiblemente|"
    r"sospecha de|no se puede descartar|suspect|à confirmer|mogelijk|verdacht voor)\b",
    re.IGNORECASE,
)
COMPILED = {
    t: {p: [re.compile(x, re.IGNORECASE) for x in pats] for p, pats in groups.items()}
    for t, groups in RULES.items()
}

def classify_report(report, target):
    text = re.sub(r"\s+", " ", str(report or ""))
    spans = {"positive": [], "negative": []}
    uncertain = False
    for polarity in ("positive", "negative"):
        for pattern in COMPILED[target][polarity]:
            for match in pattern.finditer(text):
                context = text[max(0, match.start()-80):min(len(text), match.end()+80)]
                if UNCERTAINTY.search(context):
                    uncertain = True
                    continue
                cue = match.group(0).strip()
                spans[polarity].append((match.start(), match.end(), cue))

    # A positive keyword can also occur inside an explicit negative phrase
    # (e.g. "no knee effusion"). Suppress only overlapping positive matches;
    # separate positive and negative evidence elsewhere remains a conflict.
    negative_spans = [(a, b) for a, b, _ in spans["negative"]]
    positive = [
        cue for a, b, cue in spans["positive"]
        if not any(a < neg_b and neg_a < b for neg_a, neg_b in negative_spans)
    ]
    negative = [cue for _, _, cue in spans["negative"]]
    positive = list(dict.fromkeys(positive))
    negative = list(dict.fromkeys(negative))

    if positive and negative:
        return np.nan, "conflict", " | ".join(positive), " | ".join(negative)
    if positive:
        return 1.0, "positive", " | ".join(positive), ""
    if negative:
        return 0.0, "negative", "", " | ".join(negative)
    if uncertain:
        return np.nan, "uncertain", "", ""
    return np.nan, "unknown", "", ""

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input-csv", default="/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv")
    parser.add_argument("--output-dir", default="/kaggle/working/exp007_weak_labels")
    args, _ = parser.parse_known_args()
    df = pd.read_csv(args.input_csv)
    required = {"StudyInstanceUID", "Report", *TARGETS}
    missing = sorted(required.difference(df.columns))
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    if df["StudyInstanceUID"].duplicated().any():
        raise ValueError("StudyInstanceUID must be unique")
    if df["Report"].isna().any():
        raise ValueError("Report contains missing values; inspect source first")

    # Exclude every study with any official label. Do not touch gold labels.
    has_gold = df[TARGETS].notna().any(axis=1)
    weak_source = df.loc[~has_gold, ["StudyInstanceUID", "Report"]]
    print(f"Total studies: {len(df)}")
    print(f"Studies with official labels excluded: {int(has_gold.sum())}")
    print(f"Unlabelled studies to process: {len(weak_source)}")

    wide, audit = [], []
    for study_id, report in weak_source[["StudyInstanceUID", "Report"]].itertuples(index=False, name=None):
        output = {"StudyInstanceUID": study_id}
        for target in TARGETS:
            label, status, positive_cues, negative_cues = classify_report(report, target)
            output[target] = label
            audit.append({
                "StudyInstanceUID": study_id, "target": target, "weak_label": label,
                "status": status, "positive_cues": positive_cues, "negative_cues": negative_cues,
            })
        wide.append(output)

    outdir = Path(args.output_dir)
    outdir.mkdir(parents=True, exist_ok=True)
    weak_labels = pd.DataFrame(wide, columns=["StudyInstanceUID", *TARGETS])
    audit_df = pd.DataFrame(audit)
    weak_labels.to_csv(outdir / "weak_labels.csv", index=False)
    audit_df.to_csv(outdir / "weak_label_audit.csv", index=False)
    summary = audit_df.groupby(["target", "status"], dropna=False).size().rename("studies").reset_index()
    summary.to_csv(outdir / "weak_label_summary.csv", index=False)

    coverage = audit_df.groupby("target").agg(
        known=("weak_label", lambda s: int(s.notna().sum())),
        positive=("weak_label", lambda s: int((s == 1).sum())),
        negative=("weak_label", lambda s: int((s == 0).sum())),
        unknown=("status", lambda s: int((s == "unknown").sum())),
        uncertain=("status", lambda s: int((s == "uncertain").sum())),
        conflict=("status", lambda s: int((s == "conflict").sum())),
    )
    print("\nWeak-label coverage by target:")
    print(coverage.to_string())
    print(f"\nSaved outputs to {outdir}")
    print("Audit samples and error rates before using these labels for training.")

if __name__ == "__main__":
    main()